In [ ]:
import numpy as np
from scipy.io import loadmat
from collections import Counter

# ---- adjust paths / case ----
m_path = "../pytspl/data/powerNetwork/case_ACTIVSg500TS.m"  # or wherever you put it

# ---- MATPOWER indices ----
BUS_I = 0
F_BUS = 0
T_BUS = 1
PF = 13

mat = loadmat(m_path)
branch = np.asarray(mat["branchResults"], dtype=float)  # (T,E,17)
bus = np.asarray(mat["busResults"], dtype=float)        # (T,V,13)

branch0 = branch[0]
bus0 = bus[0]

V, Cv = bus0.shape
E, Cb = branch0.shape

# map bus IDs -> 0..V-1
bus_ids = bus0[:, BUS_I].astype(int)
id2idx = {bid: i for i, bid in enumerate(bus_ids)}

# directed edges in file order
f_ids = branch0[:, F_BUS].astype(int)
t_ids = branch0[:, T_BUS].astype(int)
edges_truth = [(id2idx[fi], id2idx[ti]) for fi, ti in zip(f_ids, t_ids)]

nodes_truth = list(range(V))
pf_truth = branch0[:, PF].astype(float)

print("Truth V:", V, "E:", E, "Cv:", Cv, "Cb:", Cb)
print("First 5 truth edges:", edges_truth[:5])

Truth V: 500 E: 597 Cv: 17 Cb: 21
First 5 truth edges: [(1, 0), (43, 0), (420, 0), (3, 2), (61, 2)]


In [9]:
import sys
sys.path.insert(0, r"C:\Users\Andrei\HonorsProject\pytspl")
import pytspl

In [10]:
from pytspl import list_datasets
list_datasets()

['anaheim',
 'barcelona',
 'chicago-regional',
 'chicago-sketch',
 'goldcoast',
 'siouxfalls',
 'test_dataset',
 'winnipeg',
 'paper',
 'forex',
 'lastfm-1k-artist',
 'webkb-cornell',
 'webkb-texas',
 'webkb-wisconsin',
 'wsn',
 'matpower-case57',
 'matpower-case118',
 'matpower-case145',
 'matpower-case300',
 'matpower-ACTIVSg200',
 'matpower-ACTIVSg500']

In [15]:
import numpy as np

from pytspl import load_dataset
import os
print("Truth file:", m_path)

complex_loaded, coords_loaded, flow_loaded = load_dataset("matpower-ACTIVSg500", only_sc=True)

# ---- 1) Nodes/edges counts ----
assert len(complex_loaded.nodes) == len(nodes_truth), (len(complex_loaded.nodes), len(nodes_truth))
assert len(complex_loaded.edges) == len(edges_truth), (len(complex_loaded.edges), len(edges_truth))

# ---- 2) Exact edge list equality (including order) ----
# This is the most important check for feature alignment.
assert list(complex_loaded.edges) == list(edges_truth), "Edge ordering mismatch!"

# ---- 3) Coordinates sanity ----
assert isinstance(coords_loaded, dict)
assert len(coords_loaded) == len(complex_loaded.nodes), (len(coords_loaded), len(complex_loaded.nodes))

print("✅ Nodes/edges/coords basic checks passed.")

Truth file: ../pytspl/data/powerNetwork/case_ACTIVSg500TS.m
Generating coordinates using spring layout.
Num. of nodes: 500
Num. of edges: 597
Num. of triangles: 0
Shape: (500, 597, 0)
Max Dimension: 1
Coordinates: 500
Flow: 584
✅ Nodes/edges/coords basic checks passed.


In [16]:
node_feats = complex_loaded.get_node_features()

# node_feats should be dict-like with keys node id -> dict of features
assert isinstance(node_feats, dict)
assert len(node_feats) == V

# check a few nodes + full column equality
for i in [0, 1, 2, V-1]:
    for c in range(Cv):
        key = f"bus_col_{c}"
        assert key in node_feats[i], f"Missing {key} in node_features[{i}]"
        val_loaded = float(node_feats[i][key])
        val_truth = float(bus0[i, c])
        if not np.isclose(val_loaded, val_truth, rtol=0, atol=1e-9):
            raise AssertionError(f"Node {i} col {c}: loaded={val_loaded}, truth={val_truth}")

print("✅ Node features match busResults[0] for all columns (spot-checked).")

✅ Node features match busResults[0] for all columns (spot-checked).


In [17]:
for i in range(V):
    for c in range(Cv):
        val_loaded = float(node_feats[i][f"bus_col_{c}"])
        val_truth = float(bus0[i, c])
        if not np.isclose(val_loaded, val_truth, rtol=0, atol=1e-9):
            raise AssertionError(f"Mismatch at node {i}, col {c}")
print("✅ Node features match for ALL nodes/cols.")

✅ Node features match for ALL nodes/cols.


In [18]:
edge_feats = complex_loaded.get_edge_features()  # dict keyed by (u,v)
assert isinstance(edge_feats, dict)
assert len(edge_feats) == E, f"edge_features has {len(edge_feats)} entries, expected {E}"

# 1) Check PF flow dict matches
for j, e in enumerate(edges_truth):
    assert e in flow_loaded, f"Missing edge {e} in flow dict"
    val_loaded = float(flow_loaded[e])
    val_truth = float(pf_truth[j])
    if not np.isclose(val_loaded, val_truth, rtol=0, atol=1e-9):
        raise AssertionError(f"Flow mismatch at edge idx {j} {e}: loaded={val_loaded}, truth={val_truth}")

# 2) Check full branch table columns in edge_features
for j, e in enumerate(edges_truth):
    feats_e = edge_feats[e]
    for c in range(Cb):
        key = f"branch_col_{c}"
        assert key in feats_e, f"Missing {key} for edge {e}"
        val_loaded = float(feats_e[key])
        val_truth = float(branch0[j, c])
        if not np.isclose(val_loaded, val_truth, rtol=0, atol=1e-9):
            raise AssertionError(f"Edge {e} col {c}: loaded={val_loaded}, truth={val_truth}")

print("✅ Edge features + flow match branchResults[0] (all columns).")

AssertionError: edge_features has 584 entries, expected 597

In [19]:
from collections import Counter

cnt = Counter(edges_truth)
dups = {e:c for e,c in cnt.items() if c>1}

print("Duplicate directed pairs:", len(dups))
print("E:", E, "unique (u,v):", len(cnt), "edge_features keys:", len(edge_feats))

# If this triggers, you *did* overwrite parallel lines:
assert len(edge_feats) == E, "Parallel branches collapsed/overwritten (edge_features size != E)"
assert len(flow_loaded) == E, "Parallel branches collapsed/overwritten (flow size != E)"

Duplicate directed pairs: 13
E: 597 unique (u,v): 584 edge_features keys: 584


AssertionError: Parallel branches collapsed/overwritten (edge_features size != E)